# 04_Attention — Classificação de sentimentos com autoatenção

**Aluno: Tiago Galhardo Avelar**

Esta é uma cópia resolvida do modelo fornecido na aula. O objetivo é classificar avaliações do IMDb usando embeddings GloVe congelados e uma camada de autoatenção simplificada, apresentada em duas implementações:

1. uma versão didática com laços explícitos;
2. uma versão matricial, sem laços no método `forward`, adequada para minibatches.

O conjunto oficial de treino do IMDb é dividido de forma determinística em **20.000 exemplos de treino** e **5.000 de validação**. O conjunto oficial de teste, com 25.000 exemplos, é usado somente na avaliação final.


## 1. Configuração e reprodutibilidade

O comprimento máximo de 120 tokens limita o custo quadrático da atenção. O GloVe de 50 dimensões reduz memória e tempo sem deixar de usar embeddings pré-treinados. Todas as sementes são fixadas e as operações determinísticas do PyTorch são habilitadas.


In [1]:
import copy
import os
import random
import re
import tarfile
import time
import urllib.request
from pathlib import Path

import gensim
import gensim.downloader as api
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 123
PARAMS = {
    "max_length": 120,
    "batch_size": 64,
    "hidden_size": 64,
    "n_epochs": 3,
    "learning_rate": 1e-3,
}


def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)


set_seeds()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
print("PyTorch:", torch.__version__)
print("Gensim:", gensim.__version__)


Dispositivo: cpu
PyTorch: 2.14.0+cpu
Gensim: 4.4.0


## 2. Dataset IMDb: 20k treino / 5k validação

Os arquivos são ordenados antes da leitura e a divisão é embaralhada com uma instância local de `random.Random`, tornando a separação reproduzível. O download usa a distribuição publicada pela Stanford.


In [2]:
DATA_DIR = Path("aclImdb")
ARCHIVE = Path("aclImdb_v1.tar.gz")
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

if not DATA_DIR.exists():
    if not ARCHIVE.exists():
        print("Baixando IMDb...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    print("Extraindo IMDb...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(".")


def load_texts(folder):
    return [
        path.read_text(encoding="utf-8")
        for path in sorted(Path(folder).glob("*.txt"))
    ]


x_train_pos = load_texts(DATA_DIR / "train" / "pos")
x_train_neg = load_texts(DATA_DIR / "train" / "neg")
x_test_pos = load_texts(DATA_DIR / "test" / "pos")
x_test_neg = load_texts(DATA_DIR / "test" / "neg")

train_pairs = list(zip(
    x_train_pos + x_train_neg,
    [1] * len(x_train_pos) + [0] * len(x_train_neg),
))
random.Random(SEED).shuffle(train_pairs)

train_pairs, valid_pairs = train_pairs[:20_000], train_pairs[20_000:25_000]
test_pairs = list(zip(
    x_test_pos + x_test_neg,
    [1] * len(x_test_pos) + [0] * len(x_test_neg),
))

x_train, y_train = map(list, zip(*train_pairs))
x_valid, y_valid = map(list, zip(*valid_pairs))
x_test, y_test = map(list, zip(*test_pairs))

assert (len(x_train), len(x_valid), len(x_test)) == (20_000, 5_000, 25_000)
print("Treino:", len(x_train))
print("Validação:", len(x_valid))
print("Teste:", len(x_test))


Treino: 20000
Validação: 5000
Teste: 25000


## 3. Embeddings GloVe pré-treinados e congelados

É usada a coleção `glove-wiki-gigaword-50`, disponibilizada pelo Gensim. Acrescentamos uma linha aleatória para palavras desconhecidas (`<UNK>`) e uma linha de zeros para padding (`<PAD>`). As camadas são construídas com `Embedding.from_pretrained(..., freeze=True)`, logo os vetores GloVe **não são atualizados** durante o treino.


In [3]:
print("Carregando GloVe pré-treinado...")
glove_vectors = api.load("glove-wiki-gigaword-50")

vocab = dict(glove_vectors.key_to_index)
UNK_IDX = len(vocab)
PAD_IDX = UNK_IDX + 1
vocab["<UNK>"] = UNK_IDX
vocab["<PAD>"] = PAD_IDX

generator = torch.Generator().manual_seed(SEED)
unk_vector = torch.empty(1, glove_vectors.vector_size).uniform_(
    -0.05, 0.05, generator=generator
)
pad_vector = torch.zeros(1, glove_vectors.vector_size)
glove_tensor = torch.from_numpy(glove_vectors.vectors).float()
embeddings = torch.cat((glove_tensor, unk_vector, pad_vector), dim=0)

PARAMS["vocabulary_size"] = len(vocab)
PARAMS["padding_idx"] = PAD_IDX
PARAMS["embedding_dim"] = embeddings.shape[1]

print("Vocabulário total:", len(vocab))
print("Matriz de embeddings:", tuple(embeddings.shape))
print("Índices UNK/PAD:", UNK_IDX, PAD_IDX)


Carregando GloVe pré-treinado...


Vocabulário total: 400002
Matriz de embeddings: (400002, 50)
Índices UNK/PAD: 400000 400001


## 4. Tokenização, truncamento e padding

Todos os exemplos são convertidos para tensores de comprimento fixo. Essa representação permite que a versão matricial processe o minibatch inteiro de uma só vez.


In [4]:
TOKEN_RE = re.compile(r"\w+")


def tokenize(text):
    return [token.lower() for token in TOKEN_RE.findall(text)]


def to_token_ids(text, vocabulary, max_length, padding_idx):
    tokens = tokenize(text)[:max_length]
    token_ids = [vocabulary.get(token, vocabulary["<UNK>"]) for token in tokens]
    token_ids += [padding_idx] * (max_length - len(token_ids))
    return token_ids


class IMDbDataset(Dataset):
    def __init__(self, texts, labels, vocabulary, max_length, padding_idx):
        self.token_ids = torch.full(
            (len(texts), max_length),
            padding_idx,
            dtype=torch.long,
        )
        for index, text in enumerate(texts):
            ids = to_token_ids(text, vocabulary, max_length, padding_idx)
            self.token_ids[index] = torch.tensor(ids, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.token_ids[index], self.labels[index]


## 5. Autoatenção simplificada — implementação com laços

Para cada token de consulta $q_i$, calculamos o produto escalar com cada chave $k_j$, aplicamos `softmax`, fazemos a soma ponderada dos valores $v_j$ e, finalmente, calculamos a média das representações dos tokens válidos. Nesta versão didática, os laços percorrem exemplos, consultas, chaves e valores explicitamente.


In [5]:
class SelfAttentionLayerLoop(nn.Module):
    def __init__(self, pretrained_embeddings, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_embeddings,
            freeze=True,
            padding_idx=padding_idx,
        )
        self.padding_idx = padding_idx

    def forward(self, batch_token_ids):
        batch_outputs = []

        for token_ids in batch_token_ids:
            valid_ids = []
            for token_id in token_ids:
                if token_id.item() != self.padding_idx:
                    valid_ids.append(token_id)

            if not valid_ids:
                batch_outputs.append(
                    torch.zeros(
                        self.embedding.embedding_dim,
                        device=batch_token_ids.device,
                        dtype=self.embedding.weight.dtype,
                    )
                )
                continue

            query_outputs = []
            for query_id in valid_ids:
                query = self.embedding(query_id)
                scores = []
                values = []

                for key_id in valid_ids:
                    key = self.embedding(key_id)
                    scores.append(torch.dot(query, key))
                    values.append(key)  # Q = K = V = embeddings

                weights = torch.softmax(torch.stack(scores), dim=0)
                attended = torch.zeros_like(values[0])
                for weight, value in zip(weights, values):
                    attended = attended + weight * value
                query_outputs.append(attended)

            batch_outputs.append(torch.stack(query_outputs).mean(dim=0))

        return torch.stack(batch_outputs)


## 6. Autoatenção simplificada — implementação matricial

Não há laços no `forward`. Para um minibatch de embeddings `E` com forma `[B, L, D]`:

- `E @ Eᵀ` produz todos os escores com forma `[B, L, L]`;
- a máscara remove chaves e consultas de padding;
- `softmax` normaliza cada linha;
- uma segunda multiplicação matricial produz os vetores contextualizados;
- a média mascarada agrega apenas consultas válidas.


In [6]:
class SelfAttentionLayerMatrix(nn.Module):
    def __init__(self, pretrained_embeddings, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_embeddings,
            freeze=True,
            padding_idx=padding_idx,
        )
        self.padding_idx = padding_idx

    def forward(self, batch_token_ids):
        embeddings_batch = self.embedding(batch_token_ids)          # [B, L, D]
        valid_mask = batch_token_ids.ne(self.padding_idx)            # [B, L]

        scores = torch.bmm(                                          # [B, L, L]
            embeddings_batch,
            embeddings_batch.transpose(1, 2),
        )
        key_mask = valid_mask.unsqueeze(1)                           # [B, 1, L]
        scores = scores.masked_fill(
            ~key_mask,
            torch.finfo(scores.dtype).min,
        )

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = attention_weights * valid_mask.unsqueeze(-1)
        contextualized = torch.bmm(attention_weights, embeddings_batch)

        valid_queries = valid_mask.unsqueeze(-1)
        summed = (contextualized * valid_queries).sum(dim=1)
        counts = valid_queries.sum(dim=1).clamp_min(1)
        mean_embeddings = summed / counts
        return mean_embeddings


## 7. Teste unitário com embeddings artificiais

O teste fornecido no modelo da aula é preservado. As duas implementações devem reproduzir o mesmo resultado-alvo e também coincidir entre si.


In [7]:
fake_vocab = {"a": 0, "b": 1, "c": 2, "<UNK>": 3}
fake_embeddings = torch.arange(0, 2 * len(fake_vocab)).reshape(len(fake_vocab), 2).float()
fake_embeddings = torch.cat((fake_embeddings, torch.zeros(1, 2)), dim=0)
fake_examples = ["a", "a b", "a c b", "a z"]

fake_batch = torch.tensor([
    to_token_ids(text, fake_vocab, max_length=2, padding_idx=4)
    for text in fake_examples
])
target_output = torch.tensor([
    [0.00000000, 1.00000000],
    [1.88075161, 2.88075161],
    [3.96402740, 4.96402740],
    [5.99258232, 6.99258232],
])

loop_fake = SelfAttentionLayerLoop(fake_embeddings, padding_idx=4)
matrix_fake = SelfAttentionLayerMatrix(fake_embeddings, padding_idx=4)
loop_output = loop_fake(fake_batch)
matrix_output = matrix_fake(fake_batch)

assert torch.allclose(loop_output, target_output, atol=1e-6)
assert torch.allclose(matrix_output, target_output, atol=1e-6)
assert torch.allclose(loop_output, matrix_output, atol=1e-6)
print("Teste artificial aprovado.")
print(matrix_output)


Teste artificial aprovado.
tensor([[0.0000, 1.0000],
        [1.8808, 2.8808],
        [3.9640, 4.9640],
        [5.9926, 6.9926]])


## 8. Equivalência e eficiência nas avaliações do IMDb

Comparamos as implementações em quatro avaliações reais truncadas para 60 tokens. A medição é feita na CPU e inclui aquecimento. A saída numérica deve coincidir; espera-se que a versão matricial seja substancialmente mais rápida.


In [8]:
benchmark_length = 60
benchmark_batch = torch.tensor([
    to_token_ids(text, vocab, benchmark_length, PAD_IDX)
    for text in x_valid[:4]
])

loop_attention = SelfAttentionLayerLoop(embeddings, PAD_IDX).eval()
matrix_attention = SelfAttentionLayerMatrix(embeddings, PAD_IDX).eval()


def measure_seconds(layer, inputs, repeats):
    durations = []
    with torch.no_grad():
        layer(inputs)  # aquecimento
        for _ in range(repeats):
            start = time.perf_counter()
            layer(inputs)
            durations.append(time.perf_counter() - start)
    return float(np.median(durations))


with torch.no_grad():
    output_loop = loop_attention(benchmark_batch)
    output_matrix = matrix_attention(benchmark_batch)

assert torch.allclose(output_loop, output_matrix, rtol=1e-5, atol=1e-6)

loop_seconds = measure_seconds(loop_attention, benchmark_batch, repeats=2)
matrix_seconds = measure_seconds(matrix_attention, benchmark_batch, repeats=10)
speedup = loop_seconds / matrix_seconds

print(f"Tempo com laços: {loop_seconds:.6f} s")
print(f"Tempo matricial: {matrix_seconds:.6f} s")
print(f"Aceleração matricial: {speedup:.1f}x")
print("Maior diferença absoluta:", (output_loop - output_matrix).abs().max().item())


Tempo com laços: 0.262248 s
Tempo matricial: 0.000213 s
Aceleração matricial: 1230.6x
Maior diferença absoluta: 2.384185791015625e-07


## 9. Classificador binário com autoatenção matricial

O classificador usa apenas a implementação matricial no treinamento completo. A camada GloVe permanece congelada; somente as duas camadas lineares são treinadas.


In [9]:
class AttentionClassifier(nn.Module):
    def __init__(self, pretrained_embeddings, padding_idx, hidden_size):
        super().__init__()
        self.attention = SelfAttentionLayerMatrix(pretrained_embeddings, padding_idx)
        embedding_dim = pretrained_embeddings.shape[1]
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(hidden_size, 2),
        )

    def forward(self, token_ids):
        representation = self.attention(token_ids)
        return self.classifier(representation)


set_seeds()
model = AttentionClassifier(
    embeddings,
    padding_idx=PAD_IDX,
    hidden_size=PARAMS["hidden_size"],
).to(DEVICE)

assert model.attention.embedding.weight.requires_grad is False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(model)
print(f"Parâmetros treináveis: {trainable:,}")
print(f"Parâmetros congelados: {frozen:,}")


AttentionClassifier(
  (attention): SelfAttentionLayerMatrix(
    (embedding): Embedding(400002, 50, padding_idx=400001)
  )
  (classifier): Sequential(
    (0): Linear(in_features=50, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)
Parâmetros treináveis: 3,394
Parâmetros congelados: 20,000,100


## 10. Preparação dos minibatches

O gerador do `DataLoader` também recebe a seed fixa. O pré-processamento é feito uma única vez para que a medição do treino represente principalmente o custo do modelo.


In [10]:
print("Tokenizando treino...")
train_dataset = IMDbDataset(x_train, y_train, vocab, PARAMS["max_length"], PAD_IDX)
print("Tokenizando validação...")
valid_dataset = IMDbDataset(x_valid, y_valid, vocab, PARAMS["max_length"], PAD_IDX)
print("Tokenizando teste...")
test_dataset = IMDbDataset(x_test, y_test, vocab, PARAMS["max_length"], PAD_IDX)


def make_loader(dataset, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=PARAMS["batch_size"],
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_loader(train_dataset, shuffle=True)
valid_loader = make_loader(valid_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)
print("Minibatches:", len(train_loader), len(valid_loader), len(test_loader))


Tokenizando treino...


Tokenizando validação...


Tokenizando teste...


Minibatches: 313 79 391


## 11. Treinamento e acurácia

A perda é a entropia cruzada. A melhor configuração segundo a perda de validação é mantida em memória e restaurada antes do teste.


In [11]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for token_ids, labels in loader:
            token_ids = token_ids.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(token_ids)
            loss = criterion(logits, labels)
            loss_sum += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return {"loss": loss_sum / total, "accuracy": correct / total}


def train(model, train_loader, valid_loader):
    optimizer = torch.optim.Adam(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=PARAMS["learning_rate"],
    )
    best_state = None
    best_valid_loss = float("inf")
    history = []

    training_start = time.perf_counter()
    for epoch in range(1, PARAMS["n_epochs"] + 1):
        model.train()
        loss_sum = 0.0
        correct = 0
        total = 0
        epoch_start = time.perf_counter()

        for token_ids, labels in train_loader:
            token_ids = token_ids.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(token_ids)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_metrics = {
            "loss": loss_sum / total,
            "accuracy": correct / total,
        }
        valid_metrics = evaluate(model, valid_loader)
        elapsed = time.perf_counter() - epoch_start
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "valid_loss": valid_metrics["loss"],
            "valid_accuracy": valid_metrics["accuracy"],
            "seconds": elapsed,
        }
        history.append(row)
        print(row)

        if valid_metrics["loss"] < best_valid_loss:
            best_valid_loss = valid_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    total_seconds = time.perf_counter() - training_start
    return history, total_seconds


history, training_seconds = train(model, train_loader, valid_loader)
best_valid = evaluate(model, valid_loader)
test_metrics = evaluate(model, test_loader)

print(f"Tempo total de treino: {training_seconds:.2f} s")
print("Melhor validação:", best_valid)
print("Teste final:", test_metrics)
assert 0.0 <= best_valid["accuracy"] <= 1.0
assert 0.0 <= test_metrics["accuracy"] <= 1.0


{'epoch': 1, 'train_loss': 0.6588542767524719, 'train_accuracy': 0.6083, 'valid_loss': 0.6171872415542603, 'valid_accuracy': 0.6752, 'seconds': 2.618653200013796}


{'epoch': 2, 'train_loss': 0.6082567878723144, 'train_accuracy': 0.67435, 'valid_loss': 0.5819537930488586, 'valid_accuracy': 0.703, 'seconds': 3.3161438000097405}


{'epoch': 3, 'train_loss': 0.5881758602142334, 'train_accuracy': 0.68935, 'valid_loss': 0.573163324546814, 'valid_accuracy': 0.6998, 'seconds': 3.4152882000198588}


Tempo total de treino: 9.44 s
Melhor validação: {'loss': 0.573163324546814, 'accuracy': 0.6998}
Teste final: {'loss': 0.5854167738342285, 'accuracy': 0.69228}


## 12. Síntese dos resultados

A célula abaixo reúne as evidências principais exigidas: congelamento do GloVe, equivalência numérica das duas implementações, ganho de velocidade da forma matricial e acurácias obtidas.


In [12]:
summary = {
    "aluno": "Tiago Galhardo Avelar",
    "glove_congelado": not model.attention.embedding.weight.requires_grad,
    "diferenca_maxima_loop_vs_matriz": float((output_loop - output_matrix).abs().max()),
    "tempo_loop_segundos": loop_seconds,
    "tempo_matricial_segundos": matrix_seconds,
    "aceleracao_matricial": speedup,
    "melhor_acuracia_validacao": best_valid["accuracy"],
    "acuracia_teste": test_metrics["accuracy"],
}
summary


{'aluno': 'Tiago Galhardo Avelar',
 'glove_congelado': True,
 'diferenca_maxima_loop_vs_matriz': 2.384185791015625e-07,
 'tempo_loop_segundos': 0.2622478499979479,
 'tempo_matricial_segundos': 0.00021309999283403158,
 'aceleracao_matricial': 1230.6328428748193,
 'melhor_acuracia_validacao': 0.6998,
 'acuracia_teste': 0.69228}

## Ferramentas e referências

- **PyTorch** — tensores, `Embedding.from_pretrained`, `DataLoader`, diferenciação automática e otimização. Documentação: [Embedding.from_pretrained](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html#torch.nn.Embedding.from_pretrained) e [DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader).
- **Gensim** — download e leitura do modelo `glove-wiki-gigaword-50`. Documentação: [Gensim Downloader API](https://radimrehurek.com/gensim/downloader.html).
- **NumPy** — sementes e estatística de tempo (mediana). Documentação: [NumPy](https://numpy.org/doc/stable/).
- **Jupyter Notebook / nbclient** — execução programática e preservação das saídas no arquivo `.ipynb`. Documentação: [nbclient](https://nbclient.readthedocs.io/).
- **IMDb Large Movie Review Dataset** — Maas et al., *Learning Word Vectors for Sentiment Analysis*, ACL 2011: [artigo](https://aclanthology.org/P11-1015/) e [dataset](https://ai.stanford.edu/~amaas/data/sentiment/).
- **GloVe** — Pennington, Socher e Manning, *GloVe: Global Vectors for Word Representation*, EMNLP 2014: [artigo](https://aclanthology.org/D14-1162/).
- **Autoatenção** — Vaswani et al., *Attention Is All You Need*, NeurIPS 2017: [artigo](https://arxiv.org/abs/1706.03762). A atividade usa uma versão simplificada sem projeções aprendidas e sem o fator de escala.


## Comentários

---
**Comentários:**

Ferramenta: OpenAI Codex (modelo GPT-5)

Utilização: Utilizado para auxiliar na implementação das versões com laços e matricial da autoatenção, revisar máscaras de padding e operações em minibatches, corrigir o código-base, estruturar os testes de equivalência e eficiência, documentar referências e apoiar a execução reprodutível do notebook. Os resultados numéricos foram produzidos pela execução efetiva das células e verificados por asserções.

Ferramenta: PyTorch

Utilização: Utilizado para embeddings congelados, operações tensoriais, treinamento, avaliação e medição da acurácia.

Ferramenta: Gensim

Utilização: Utilizado para obter os embeddings pré-treinados GloVe `glove-wiki-gigaword-50`.

Ferramenta: Jupyter Notebook / nbclient

Utilização: Utilizado para executar todas as células e registrar as evidências no arquivo entregue.


## Conclusão

As duas camadas implementam a mesma autoatenção simplificada e produzem saídas numericamente equivalentes. A formulação matricial aproveita multiplicações em lote e elimina laços Python do `forward`, sendo a escolha adequada para treinar com 20 mil avaliações. Os embeddings GloVe permaneceram congelados durante todo o processo, enquanto o classificador aprendeu a tarefa binária a partir das representações contextualizadas.
